# Phase 1 — Data Loading & Preparation
**Goal:** Load all AWV fietstellingen data, merge the three datasets, filter for cyclists, engineer temporal features, and save a clean processed file.

> Run `download_data.py` first to populate `data/raw/` before executing this notebook.

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
import holidays
import warnings
warnings.filterwarnings('ignore')

RAW_DIR     = Path('../data/raw')
COUNTS_DIR  = RAW_DIR / 'counts'
PROC_DIR    = Path('../data/processed')
PROC_DIR.mkdir(parents=True, exist_ok=True)

print('Raw data folder exists:', RAW_DIR.exists())
print('Number of monthly count files found:',
      len(list(COUNTS_DIR.glob('data-*.csv'))))

Raw data folder exists: True
Number of monthly count files found: 72


## 1. Load Metadata (sites + directions)

In [6]:
sites = pd.read_csv(
    RAW_DIR / 'sites.csv',
    sep=',',
    header=None,
    names=['site_id', 'site_nr', 'long', 'lat', 'naam', 'domein',
           'wegnr', 'district', 'gemeente', 'interval', 'datum_van']
)
print(f'sites → {sites.shape[0]} rows')
sites.head(3)

sites → 151 rows


,site_id,site_nr,long,lat,naam,domein,wegnr,district,gemeente,interval,datum_van
0,1,100046096,4.456122,50.916183,Machelen,Vlaamse Overheid A. Wegen enVerkeer,T2110002,AWV212,Machelen,15,2019-08-22
1,2,100052862,4.471690,51.275120,Brasschaat 2,Vlaamse Overheid A. Wegen enVerkeer,N0010002,AWV123,Brasschaat,15,2019-08-22
2,3,100052863,4.472220,51.275030,Brasschaat 1,Vlaamse Overheid A. Wegen enVerkeer,N0010001,AWV123,Brasschaat,15,2019-08-22


In [7]:

richtingen = pd.read_csv(
    RAW_DIR / 'richtingen.csv',
    sep=',',
    header=None,
    names=['site_id', 'richting', 'naam'],
    engine='python',        # ← add this
    on_bad_lines='skip'     # ← add this
)
print(f'richtingen → {richtingen.shape}')
richtingen.head(3)

richtingen → (305, 3)


,site_id,richting,naam
0,1,IN,Machelen Cyclists rich. Brucargo
1,1,OUT,Machelen Cyclists richting Machelen
2,2,IN,Brasschaat 2 Fietsers rich Merksem


## 2. Load & Concatenate Monthly Count Files (2020–2025)

In [8]:
count_files = sorted(COUNTS_DIR.glob('data-*.csv'))
print(f'Found {len(count_files)} monthly files')

chunks = []
for fp in tqdm(count_files, desc='Reading CSVs'):
    try:
        df = pd.read_csv(
            fp,
            sep=',',
            header=None,
            names=['site_id', 'richting', 'type', 'van', 'tot', 'aantal'],
            low_memory=False
        )
        chunks.append(df)
    except Exception as e:
        print(f'  [err] {fp.name}: {e}')

counts_raw = pd.concat(chunks, ignore_index=True)
print(f'Total rows: {counts_raw.shape[0]:,}')
counts_raw.head(3)

Found 72 monthly files


Reading CSVs:   0%|          | 0/72 [00:00<?, ?it/s]

Total rows: 38,659,000


,site_id,richting,type,van,tot,aantal
0,23,IN,FIETSERS,2020-01-03 18:30:00.0,2020-01-03 18:45:00.0,0.0
1,8,IN,FIETSERS,2020-01-04 01:30:00.0,2020-01-04 01:45:00.0,0.0
2,14,IN,FIETSERS,2020-01-04 14:30:00.0,2020-01-04 14:45:00.0,2.0


## 3. Filter for Cyclists (`type = FIETSERS`)

In [9]:
print('Unique type values:')
print(counts_raw['type'].value_counts())

counts = counts_raw[counts_raw['type'].str.upper() == 'FIETSERS'].copy()
print(f'Rows after filtering for FIETSERS: {len(counts):,}')
print(f'Rows dropped: {len(counts_raw) - len(counts):,}')

Unique type values:
type
FIETSERS       38641714
VOETGANGERS       17286
Name: count, dtype: int64
Rows after filtering for FIETSERS: 38,641,714
Rows dropped: 17,286


## 4. Parse Timestamps & Remove Duplicates

In [10]:
counts['van'] = pd.to_datetime(counts['van'], errors='coerce')
counts['tot'] = pd.to_datetime(counts['tot'], errors='coerce')

n_before = len(counts)
counts.dropna(subset=['van', 'tot', 'aantal'], inplace=True)
counts['aantal'] = pd.to_numeric(counts['aantal'], errors='coerce')
counts.dropna(subset=['aantal'], inplace=True)
counts = counts[counts['aantal'] >= 0]
counts.drop_duplicates(subset=['site_id', 'richting', 'van'], inplace=True)

print(f'Rows before cleaning : {n_before:,}')
print(f'Rows after  cleaning : {len(counts):,}')
print(f'Date range: {counts["van"].min()} → {counts["van"].max()}')

Rows before cleaning : 38,641,714
Rows after  cleaning : 37,634,223
Date range: 2020-01-01 00:00:00 → 2025-12-31 23:45:00


## 5. Merge with Sites & Directions Metadata

In [11]:
for df in [sites, richtingen]:
    for col in df.columns:
        if 'site' in col and 'id' in col:
            df.rename(columns={col: 'site_id'}, inplace=True)
            break

df = counts.merge(sites, on='site_id', how='left', suffixes=('', '_site'))
df = df.merge(richtingen, on=['site_id', 'richting'], how='left', suffixes=('', '_dir'))
print('Merged dataframe shape:', df.shape)

Merged dataframe shape: (37634223, 17)


## 6. Temporal Feature Engineering

In [12]:
be_holidays = holidays.Belgium(years=range(2020, 2026))

df['hour']       = df['van'].dt.hour
df['day_of_week']= df['van'].dt.dayofweek
df['day_name']   = df['van'].dt.day_name()
df['month']      = df['van'].dt.month
df['year']       = df['van'].dt.year
df['date']       = df['van'].dt.date
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
df['is_holiday'] = df['date'].astype(str).map(lambda d: 1 if d in be_holidays else 0)

def get_season(month):
    if month in [12, 1, 2]:  return 'Winter'
    elif month in [3, 4, 5]: return 'Spring'
    elif month in [6, 7, 8]: return 'Summer'
    else:                     return 'Autumn'

df['season'] = df['month'].map(get_season)
print(df[['van','hour','day_of_week','month','year','is_weekend','is_holiday','season']].head(10))

                  van  hour  day_of_week  month  year  is_weekend  is_holiday  \
0 2020-01-03 18:30:00    18            4      1  2020           0           0   
1 2020-01-04 01:30:00     1            5      1  2020           1           0   
2 2020-01-04 14:30:00    14            5      1  2020           1           0   
3 2020-01-03 02:45:00     2            4      1  2020           0           0   
4 2020-01-03 03:30:00     3            4      1  2020           0           0   
5 2020-01-04 15:15:00    15            5      1  2020           1           0   
6 2020-01-03 20:30:00    20            4      1  2020           0           0   
7 2020-01-03 03:15:00     3            4      1  2020           0           0   
8 2020-01-03 15:15:00    15            4      1  2020           0           0   
9 2020-01-03 14:15:00    14            4      1  2020           0           0   

   season  
0  Winter  
1  Winter  
2  Winter  
3  Winter  
4  Winter  
5  Winter  
6  Winter  
7  Winter  


In [13]:
df.sort_values(['site_id', 'van'], inplace=True)
df['rolling_7d_avg'] = (
    df.groupby('site_id')['aantal']
    .transform(lambda x: x.rolling(window=7*24, min_periods=1).mean())
)
print('Rolling average computed.')

Rolling average computed.


## 7. Data Quality Assessment

In [14]:
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0])

site_counts = df.groupby('site_id')['aantal'].count()
expected = 6 * 365 * 24
completeness = (site_counts / expected).clip(upper=1.0)
good_sites = completeness[completeness >= 0.80].index
print(f'Sites with >= 80% completeness: {len(good_sites)} / {len(site_counts)}')

df_clean = df[df['site_id'].isin(good_sites)].copy()
print(f'Rows after quality filter: {len(df_clean):,}')

=== Missing Values ===
wegnr       170480
district    170480
gemeente     24286
dtype: int64
Sites with >= 80% completeness: 141 / 149
Rows after quality filter: 37,488,307


## 8. Save Processed Data

In [15]:
out_path = PROC_DIR / 'cyclists_clean.parquet'
df_clean.to_parquet(out_path, index=False)
print(f'Saved to: {out_path}')
print(f'Final shape: {df_clean.shape}')

Saved to: ..\data\processed\cyclists_clean.parquet
Final shape: (37488307, 27)


## ✅ Phase 1 Complete
The processed file `data/processed/cyclists_clean.parquet` is ready for:
- `02_eda.ipynb` — Exploratory visualisations
- `03_clustering.ipynb` — Site classification
- `04_weather_modeling.ipynb` — Weather sensitivity modelling